In [0]:
import pandas as pd
import numpy as np

print("Libraries loaded successfully")

In [0]:
DATA_PATH = "demand_forecasting.csv"

df = pd.read_csv(DATA_PATH)

print("Original rows:", len(df))
print("Original columns:", len(df.columns))

display(df.head())

In [0]:
df["Date"] = pd.to_datetime(df["Date"])

df = df.sort_values(
    ["Store ID", "Product ID", "Date"]
).reset_index(drop=True)

print("Data sorted successfully")

display(
    df[
        [
            "Date",
            "Store ID",
            "Product ID",
            "Demand"
        ]
    ].head(20)
)

In [0]:
GROUP_COLS = ["Store ID","Product ID"]

print("Time-series grouping columns:", GROUP_COLS)

In [0]:
df["day_of_week"] = df["Date"].dt.dayofweek
df["day_of_month"] = df["Date"].dt.day
df["week_of_year"] = df["Date"].dt.isocalendar().week.astype(int)
df["month"] = df["Date"].dt.month
df["quarter"] = df["Date"].dt.quarter
df["year"] = df["Date"].dt.year

df["is_weekend"] = (
    df["day_of_week"] >= 5
).astype(int)

display(
    df[
        [
            "Date",
            "day_of_week",
            "month",
            "quarter",
            "year",
            "is_weekend"
        ]
    ].head()
)

In [0]:
GROUP_COLS = ["Store ID", "Product ID"]

df["lag_1"] = (df.groupby(GROUP_COLS)["Demand"].shift(1))

df["lag_7"] = (df.groupby(GROUP_COLS)["Demand"].shift(7))

df["lag_14"] = (df.groupby(GROUP_COLS)["Demand"].shift(14))

df["lag_28"] = (df.groupby(GROUP_COLS)["Demand"].shift(28))

display(
    df[
        [
            "Date",
            "Store ID",
            "Product ID",
            "Demand",
            "lag_1",
            "lag_7",
            "lag_14",
            "lag_28"
        ]
    ].head(35)
)

In [0]:
df["rolling_mean_7"] = (
    df.groupby(GROUP_COLS)["Demand"]
      .transform(
          lambda x: x.shift(1).rolling(7).mean()
      )
)

df["rolling_mean_14"] = (
    df.groupby(GROUP_COLS)["Demand"]
      .transform(
          lambda x: x.shift(1).rolling(14).mean()
      )
)

df["rolling_mean_28"] = (
    df.groupby(GROUP_COLS)["Demand"]
      .transform(
          lambda x: x.shift(1).rolling(28).mean()
      )
)

In [0]:
df["rolling_std_7"] = (
    df.groupby(GROUP_COLS)["Demand"]
      .transform(
          lambda x: x.shift(1).rolling(7).std()
      )
)

df["rolling_std_14"] = (
    df.groupby(GROUP_COLS)["Demand"]
      .transform(
          lambda x: x.shift(1).rolling(14).std()
      )
)

In [0]:
feature_columns = [
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28",
    "rolling_std_7",
    "rolling_std_14"
]

display(
    df[feature_columns]
    .isnull()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_values")
)


In [0]:
df_model = df.dropna(
    subset=[
        "lag_1",
        "lag_7",
        "lag_14",
        "lag_28",
        "rolling_mean_7",
        "rolling_mean_14",
        "rolling_mean_28"
    ]
).copy()

print("Original rows:", len(df))
print("Model rows:", len(df_model))

In [0]:
CATEGORICAL_FEATURES = [
    "Store ID",
    "Product ID",
    "Category",
    "Region",
    "Weather Condition",
    "Seasonality",
    "Epidemic"
]

print("Categorical features:")
print(CATEGORICAL_FEATURES)

In [0]:
BUSINESS_FEATURES = [
    "Inventory Level",
    "Units Ordered",
    "Price",
    "Discount",
    "Promotion",
    "Competitor Pricing"
]

print("Business features:")
print(BUSINESS_FEATURES)


In [0]:
TIME_FEATURES = [
    "day_of_week",
    "day_of_month",
    "week_of_year",
    "month",
    "quarter",
    "year",
    "is_weekend"
]

print("Time features:")
print(TIME_FEATURES)

In [0]:
LAG_FEATURES = [
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28"
]

print("Lag features:")
print(LAG_FEATURES)

In [0]:
ROLLING_FEATURES = [
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28",
    "rolling_std_7",
    "rolling_std_14"
]

print("Rolling features:")
print(ROLLING_FEATURES)


In [0]:
FEATURES = (
    LAG_FEATURES
    + ROLLING_FEATURES
    + BUSINESS_FEATURES
    + TIME_FEATURES
    + CATEGORICAL_FEATURES
)

TARGET = "Demand"

print("Number of features:", len(FEATURES))

print("\nFeatures:")
for feature in FEATURES:
    print("-", feature)

print("\nTarget:", TARGET)

In [0]:
missing_features = [
    feature
    for feature in FEATURES
    if feature not in df_model.columns
]

if len(missing_features) == 0:
    print("All features exist successfully")
else:
    print("Missing features:")
    print(missing_features)


In [0]:
model_columns = (
    ["Date"]
    + GROUP_COLS
    + FEATURES
    + [TARGET]
)

df_model = df_model[
    model_columns
].copy()

print(
    "Final model dataset shape:",
    df_model.shape
)

display(df_model.head())


In [0]:
for col in CATEGORICAL_FEATURES:

    df_model[col] = (
        df_model[col]
        .astype("category")
    )

print("Categorical columns converted successfully")


In [0]:
print("====================================")
print("FEATURE ENGINEERING SUMMARY")
print("====================================")

print("Final rows      :", len(df_model))
print("Final columns   :", len(df_model.columns))
print("Number features :", len(FEATURES))
print("Target          :", TARGET)

print(
    "Remaining nulls :",
    df_model[FEATURES + [TARGET]]
    .isnull()
    .sum()
    .sum()
)

print("====================================")

In [0]:
display(
    df_model.head(20)
)


# COMMAND ----------
# 22. Feature engineering completed

print("Feature engineering completed successfully.")